# Customer RFM Segmentation

Reads `retail.orders` (see `data-generators/retail_order_ingest.ipynb`)
and scores each customer on Recency / Frequency / Monetary value — the
standard lifecycle-marketing segmentation used to target win-back offers,
loyalty rewards, and at-risk outreach.

Cleans out cancelled orders (`invoice_no` prefixed `C`) and non-positive
quantity/price rows first, then scores each customer into quintiles
(1-5) per RFM dimension and buckets the combined score into a named
segment.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "retail"
SOURCE_TABLE = "orders"
TARGET_TABLE = "customer_rfm_segments_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Clean: drop cancellations and non-positive rows

In [ ]:
clean_df = (
    df
    .filter(~F.col("invoice_no").cast("string").startswith("C"))
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0)
)

max_invoice_date = clean_df.agg(F.max("invoice_date")).collect()[0][0]

## Transform: recency / frequency / monetary per customer

In [ ]:
rfm_df = (
    clean_df
    .groupBy("customer_id")
    .agg(
        F.datediff(F.lit(max_invoice_date), F.max("invoice_date")).alias("recency_days"),
        F.countDistinct("invoice_no").alias("frequency"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("monetary"),
        F.first("country").alias("country"),
    )
)

## Score into quintiles and label segments

Recency: more recent = higher score (ntile is inverted). Frequency and
monetary: higher = higher score. Combined score (3-15) maps to a named
segment.

In [ ]:
recency_window = Window.orderBy(F.asc("recency_days"))
frequency_window = Window.orderBy(F.asc("frequency"))
monetary_window = Window.orderBy(F.asc("monetary"))

scored_df = (
    rfm_df
    .withColumn("recency_score", 6 - F.ntile(5).over(recency_window))
    .withColumn("frequency_score", F.ntile(5).over(frequency_window))
    .withColumn("monetary_score", F.ntile(5).over(monetary_window))
)

summary_df = (
    scored_df
    .withColumn("rfm_score", F.col("recency_score") + F.col("frequency_score") + F.col("monetary_score"))
    .withColumn(
        "segment",
        F.when(F.col("rfm_score") >= 13, "Champions")
        .when(F.col("rfm_score") >= 10, "Loyal Customers")
        .when(F.col("rfm_score") >= 7, "Potential Loyalists")
        .when(F.col("rfm_score") >= 4, "At Risk")
        .otherwise("Lost"),
    )
    .orderBy(F.desc("rfm_score"))
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_customers FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT segment, COUNT(*) AS customers FROM {DATABASE_NAME}.{TARGET_TABLE} GROUP BY segment ORDER BY customers DESC").show()